# Rung-1 comparison: le2, lo2

Tasks: fastapi_15589,fastapi_13920,fastapi_14258,fastapi_11194,fastapi_14487,fastapi_15280,fastapi_14419,fastapi_15800,fastapi_14306,rich_3882,rich_3470,rich_4079,rich_3521,rich_3934,rich_3468,rich_3180

Generated by scripts/make_kaggle_notebook.py. Attach competition data, the wheelhouse dataset and the 31B model; accelerator L4 ×4; Run All; stop the session when done.

In [ ]:
# 1. Install the competition wheelhouse BEFORE anything imports google.adk
#    (Kaggle preinstalls another google-adk; importing it first causes pydantic class clashes).
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not WHEELHOUSE_DIR.exists():
    WHEELHOUSE_DIR = Path(glob.glob('/kaggle/input/**/gemma-4-developer-agent-wheelhouse', recursive=True)[0])

for pth in glob.glob('/usr/local/lib/python*/*-packages/*cutlass*.pth'):
    try:
        os.unlink(pth)
    except OSError:
        pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target = tmp_whl / (w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name)
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
assert wheels, f'No wheels found in {WHEELHOUSE_DIR}'
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

In [ ]:
# 2. Write the submissions to evaluate
import re
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
SUB_ROOT = WORKING_DIR / 'submissions'
TASK_IDS = ['fastapi_15589', 'fastapi_13920', 'fastapi_14258', 'fastapi_11194', 'fastapi_14487', 'fastapi_15280', 'fastapi_14419', 'fastapi_15800', 'fastapi_14306', 'rich_3882', 'rich_3470', 'rich_4079', 'rich_3521', 'rich_3934', 'rich_3468', 'rich_3180']
CONFIGS = ['le2', 'lo2']
REPEATS = 1
CONCURRENCY = 2  # tasks evaluated at the same time (each has its own workspace and /tmp)
EMBEDDED = {
 "le2": {
  "agent.yaml": "# Arm le2 (rung-3f): le with the fixed scout parser (accepts every argument shape, default line ranges). le: locator -> editor. The locator (deep thinking, low temperature) reads only through the scout\n# skill (16 reading calls, then it refuses), writes /tmp/plan.md, calls the editor, checks the diff + a test, allows\n# one correction, submits. The editor (light settings) applies the plan and answers EDITED/FAILED.\nname: locator\ndescription: Finds what to change, plans it for the editor, checks the result and submits.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/locator.md\ntools:\n  - get_status\n  - submit_patch\n  - agent_tool:\n      config_path: sub_agents/editor.yaml\nskills:\n  - skills/scout\ngenerate_content_config: !include configs/locator_sampling.yaml\n",
  "configs/editor_sampling.yaml": "# Editor (rung-3e le): r4 settings halved (thinking 512 -> 256, max_output 2048 -> 1024); it only applies a plan.\ntemperature: 1.0\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 1024\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 256\n",
  "configs/locator_sampling.yaml": "# Locator (rung-3e le): low temperature + deep thinking (budget to be set from rung-3d: 4096 / 2048 / 1024).\ntemperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 2048\n",
  "eval_config.yaml": "# Arm le2 (rung-3f): le with the fixed scout parser (accepts every argument shape, default line ranges). le: 36 calls (helper calls share the budget), 8 min, 90 LLM turns across both agents.\nevaluation:\n  timeout_seconds: 240\n  max_tool_calls: 36\n  max_time_minutes: 8\n  max_turns: 90\n",
  "prompts/editor.md": "You are the editor. A locator has saved a plan in /tmp/plan.md. Apply it exactly; do not explore, redesign or test.\n\n1. Read the plan: run_command with cat /tmp/plan.md\n2. For each FILE and CHANGE: call edit_file with old_string = the exact old lines from the plan and new_string = the\n   new lines. If edit_file returns an error, read the real lines once (read_file with start_line and end_line around\n   them) and retry with the text exactly as it is in the file, or edit with run_command:\n   python3 - << 'EOF'\n   p = 'path/to/file.py'\n   s = open(p).read()\n   old = '''exact old lines'''\n   new = '''new lines'''\n   assert s.count(old) == 1\n   open(p, 'w').write(s.replace(old, new))\n   EOF\n3. Reply with one line per file and nothing else:\n   EDITED: path/to/file.py: what changed\n   FAILED: path/to/file.py: why (quote the error)\n\nNever edit tests, conftest.py or configuration files. Call tools only through the tool-calling interface.\n",
  "prompts/locator.md": "You are the locator in a two-agent team fixing one task in the repository at /workspace. You find what must\nchange, write a plan, and check the result; the editor makes the edit. Nobody will answer questions: decide from\nthe task text, the code, the docs and the tests.\n\n## The task (verbatim)\n{problem_description}\n\n## Your tools\n- scout is your only way to read code. Call run_skill_script with skill_name scout, file_path scripts/scout.py and:\n  find:  args {\"mode\": \"find\", \"query\": \"exact name or message\"}   search + the code that follows each hit\n  show:  args {\"mode\": \"show\", \"file\": \"pkg/module.py\", \"start\": 40, \"end\": 170}   up to 150 numbered lines\n  history: args {\"mode\": \"history\", \"query\": \"NAME\"}   commits that added or removed NAME (useful when the task\n         is only a title or a link), or {\"mode\": \"history\", \"commit\": \"abc1234\"} for one commit's change\n  plan:  args {\"mode\": \"plan\", \"text\": \"...\"}   saves the plan for the editor\n  check: args {\"mode\": \"check\", \"test\": \"tests/test_x.py\", \"k\": \"name\"}   current diff + that test's result\n  find, show and history share 16 reading calls; plan and check are free.\n- editor: call it with the request \"Apply /tmp/plan.md\". It edits and answers EDITED or FAILED.\n- submit_patch when the change is in.\n\n## Steps\n1. From the task, note the exact names, parameters, error messages, status codes and output it asks for. The\n   hidden tests use exactly these.\n2. Read: find the most specific name or message, then show whole functions. For a new feature, read the closest\n   existing sibling feature and copy its shape (signature, imports, exports, registration). For a documentation\n   example task, look under docs_src/ and docs/.\n3. As soon as you know the change, save the plan (at the latest when scout says no reading calls are left).\n   For each file:\n   FILE: path/to/file.py\n   WHAT IT DOES: one line about the function you change\n   CHANGE: replace these exact lines, copied from show with their indentation:\n   <old lines>\n   with:\n   <new lines>\n   CONTRACT: the exact names, messages or values the tests expect\n4. Call the editor: \"Apply /tmp/plan.md\".\n5. Check with the nearest test file. Correct the plan once and call the editor once more only if the diff is not\n   what you planned, the editor answered FAILED, or a test fails because of an error in the new code. A failing\n   test that asserts the old behaviour the task asks to change is expected: keep the change. Then submit_patch.\n\n## Rules\n- When a tool result says only a few tool calls remain: if no edit is in yet, save the plan and call the editor\n  now; if it is in, call submit_patch.\n- Call tools only through the tool-calling interface; never write a tool call or its output as text.\n- Never ask to change tests; the grader resets them.\n",
  "skills/scout/SKILL.md": "---\nname: scout\ndescription: The locator's reading tool. find, show and history read the repository (16 reading calls in total); plan writes /tmp/plan.md for the editor; check shows the current diff and runs a targeted test.\n---\n\nCall it with run_skill_script, skill_name scout, file_path scripts/scout.py and named args:\n\n- find:  args = {\"mode\": \"find\", \"query\": \"convert_underscores\"}\n  git grep hits for the name or message, each with the 20 lines that follow.\n- show:  args = {\"mode\": \"show\", \"file\": \"pkg/module.py\", \"start\": 40, \"end\": 170}\n  numbered lines, up to 150 at once.\n- history: args = {\"mode\": \"history\", \"query\": \"NAME\"} (commits that added or removed it, plus the 8 latest)\n  or {\"mode\": \"history\", \"commit\": \"abc1234\"} (that commit's change).\n- plan:  args = {\"mode\": \"plan\", \"text\": \"FILE: ...\\nWHAT IT DOES: ...\\nCHANGE: ...\\nCONTRACT: ...\"}\n  saves the plan for the editor to /tmp/plan.md.\n- check: args = {\"mode\": \"check\", \"test\": \"tests/test_x.py\", \"k\": \"name\"}\n  the current git diff, plus the last lines of that test run (test and k are optional).\n\nfind, show and history share a budget of 16 reading calls; every result ends with the count. After 16 they refuse:\nwrite the plan and call the editor. plan and check never count.\n",
  "skills/scout/scripts/scout.py": "\"\"\"Scout: the locator's only way to read the repository, with a built-in reading budget.\n\nCalled through run_skill_script with named args, e.g. args={\"mode\": \"find\", \"query\": \"convert_underscores\"}.\nModes:\n  find  --query TEXT                     git grep hits with surrounding code (counts as a reading call)\n  show  --file PATH --start N --end M    numbered lines, at most 150 (counts as a reading call)\n  plan  --text TEXT                      write the plan for the editor to /tmp/plan.md (free)\n  history --query TEXT | --commit SHA    commits that added/removed TEXT, or one commit's change (counts)\n  check [--test PATH] [--k EXPR]         git diff of the current changes + optional targeted pytest (free)\nAfter LIMIT reading calls, find/show refuse and tell the locator to write the plan and call the editor.\n\"\"\"\nimport argparse\nimport ast\nimport inspect\nimport json\nimport os\nimport subprocess\nimport sys\nimport tempfile\n\nLIMIT = 16\nWARN_FROM = 13\nTMP = tempfile.gettempdir()          # the task's /tmp (per-task directory in the subprocess sandbox)\nCOUNTER = os.path.join(TMP, \"scout_reads\")\nPLAN = os.path.join(TMP, \"plan.md\")\n\n\ndef workspace():\n    \"\"\"The repository root. The skill wrapper chdirs into a temp dir, so look for the directory it started in.\"\"\"\n    for frame in inspect.stack():\n        cwd = frame.frame.f_locals.get(\"_orig_cwd\")\n        if isinstance(cwd, str) and os.path.isdir(os.path.join(cwd, \".git\")):\n            return cwd\n    for cand in (os.environ.get(\"PYTHONPATH\", \"\").split(os.pathsep)[0], \"/workspace\"):\n        if cand and os.path.isdir(os.path.join(cand, \".git\")):\n            return cand\n    sys.exit(\"scout: repository not found\")\n\n\ndef run(cmd, cwd, timeout=120):\n    try:\n        res = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True, timeout=timeout)\n        return (res.stdout + res.stderr).rstrip()\n    except subprocess.TimeoutExpired:\n        return f\"(timed out after {timeout}s)\"\n\n\ndef reading_budget():\n    \"\"\"Increment and return the number of reading calls, or refuse when the budget is spent.\"\"\"\n    used = int(open(COUNTER).read() or 0) if os.path.exists(COUNTER) else 0\n    if used >= LIMIT:\n        print(f\"READING LIMIT REACHED ({LIMIT} reading calls used). Write your plan now with mode plan, then call the \"\n              \"editor. You can still use mode check after the editor has answered.\")\n        sys.exit(0)\n    used += 1\n    with open(COUNTER, \"w\") as f:\n        f.write(str(used))\n    return used\n\n\ndef footer(used):\n    left = LIMIT - used\n    msg = f\"[reading {used} of {LIMIT} used\"\n    if used >= WARN_FROM:\n        msg += f\"; {left} left: start writing the plan\" if left else \"; none left: write the plan now\"\n    return msg + \"]\"\n\n\nMODES = (\"find\", \"show\", \"history\", \"plan\", \"check\")\nALIASES = {\"path\": \"file\", \"filename\": \"file\", \"file_path\": \"file\", \"pattern\": \"query\", \"name\": \"query\",\n           \"q\": \"query\", \"search\": \"query\", \"line\": \"start\", \"start_line\": \"start\", \"from\": \"start\",\n           \"end_line\": \"end\", \"to\": \"end\", \"sha\": \"commit\", \"test_file\": \"test\", \"content\": \"text\", \"plan\": \"text\"}\n\n\ndef literal(text):\n    \"\"\"A dict or list written as JSON or as a Python repr (ADK passes a nested dict as str(dict)), else None.\"\"\"\n    text = text.strip()\n    if not text[:1] in \"{[\":\n        return None\n    for load in (json.loads, ast.literal_eval):\n        try:\n            return load(text)\n        except (ValueError, SyntaxError):\n            pass\n    return None\n\n\ndef as_int(value):\n    try:\n        return int(float(str(value).strip()))\n    except ValueError:\n        return None\n\n\ndef parse(argv):\n    \"\"\"Read every argument shape the model produces: {\"mode\": ...} (--mode X), a nested {\"args\": {...}}, a list of\n    [\"--mode\", \"show\", ...] or [\"--mode=show\"], positional [\"show\", \"pkg/a.py\", \"40\", \"160\"], \"file:line\" and \"40-160\".\n    Missing or zero line numbers get defaults instead of an error.\"\"\"\n    opts, pos, tokens = {}, [], list(argv)\n    while tokens:\n        tok = tokens.pop(0)\n        if tok == \"--\":\n            continue\n        val = literal(tok)\n        if isinstance(val, dict):\n            opts.update({str(k).lstrip(\"-\"): v for k, v in val.items()})\n        elif isinstance(val, list):\n            tokens = [str(v) for v in val] + tokens\n        elif tok.startswith(\"--\") or (tok.startswith(\"-\") and len(tok) > 1 and not tok[1].isdigit()):\n            key, eq, value = tok.lstrip(\"-\").partition(\"=\")\n            if not eq:\n                value = tokens.pop(0) if tokens and not tokens[0].startswith(\"--\") else \"\"\n            inner = literal(value) if key == \"args\" else None\n            if isinstance(inner, dict):\n                opts.update({str(k).lstrip(\"-\"): v for k, v in inner.items()})\n            elif isinstance(inner, list):\n                tokens = [str(v) for v in inner] + tokens\n            else:\n                opts[key] = value\n        else:\n            pos.append(tok)\n    opts = {ALIASES.get(k, k): v for k, v in opts.items()}\n    if \"args\" in opts and isinstance(opts[\"args\"], dict):\n        opts.update(opts.pop(\"args\"))\n    mode = str(opts.get(\"mode\", \"\")).strip().lower()\n    if mode not in MODES and pos and pos[0].lower() in MODES:\n        mode = pos.pop(0).lower()\n    if mode not in MODES:  # infer the mode from what was given\n        mode = \"show\" if opts.get(\"file\") else \"plan\" if opts.get(\"text\") else \"history\" if opts.get(\"commit\") else \\\n            \"find\" if opts.get(\"query\") else \"check\" if opts.get(\"test\") or opts.get(\"k\") else \"\"\n    a = argparse.Namespace(mode=mode, **{k: opts.get(k) for k in (\"query\", \"file\", \"start\", \"end\", \"text\",\n                                                                    \"commit\", \"test\", \"k\")})\n    if mode == \"show\":\n        if not a.file and pos:\n            a.file = pos.pop(0)\n        if pos and a.start is None:\n            a.start = pos.pop(0)\n        if pos and a.end is None:\n            a.end = pos.pop(0)\n        if a.file and \":\" in str(a.file):  # pkg/a.py:120 or pkg/a.py:100-160\n            a.file, _, where = str(a.file).partition(\":\")\n            a.start = a.start if a.start is not None else where\n        if a.start is not None and \"-\" in str(a.start).strip(\"-\") and a.end is None:\n            a.start, _, a.end = str(a.start).partition(\"-\")\n        a.start, a.end = as_int(a.start) if a.start is not None else None, as_int(a.end) if a.end is not None else None\n        if a.file:\n            a.file = str(a.file).strip().removeprefix(\"/workspace/\").removeprefix(\"./\")\n    elif mode in (\"find\", \"history\"):\n        if not a.query and not a.commit and pos:\n            a.query = \" \".join(pos)\n        if mode == \"history\" and a.query and not a.commit and all(c in \"0123456789abcdef\" for c in a.query) \\\n                and 7 <= len(a.query) <= 40:\n            a.commit, a.query = a.query, None\n    elif mode == \"plan\":\n        if not a.text and pos:\n            a.text = \" \".join(pos)\n    elif mode == \"check\":\n        if not a.test and pos:\n            a.test = pos.pop(0)\n    return a\n\n\ndef main():\n    a = parse(sys.argv[1:])\n    if not a.mode:\n        sys.exit('scout needs a mode: args={\"mode\": \"find\", \"query\": \"NAME\"} or {\"mode\": \"show\", \"file\": \"pkg/a.py\", '\n                 '\"start\": 1, \"end\": 150}; also history, plan, check')\n    ws = workspace()\n\n    if a.mode == \"find\":\n        if not a.query:\n            sys.exit('scout find needs args={\"mode\": \"find\", \"query\": \"NAME or message\"}')\n        used = reading_budget()\n        q = a.query.replace(\"'\", \"'\\\\''\")\n        out = run(f\"git grep -n -I -A 20 -e '{q}' -- '*.py' '*.md' | head -80\", ws)\n        if not out:\n            out = run(f\"git grep -n -I -i -e '{q}' -- . | head -30\", ws) or \"(no match)\"\n        print(out)\n        print(footer(used))\n    elif a.mode == \"show\":\n        if not a.file:\n            sys.exit('scout show needs args={\"mode\": \"show\", \"file\": \"path.py\", \"start\": 40, \"end\": 160}')\n        used = reading_budget()\n        start = max(a.start or 1, 1)\n        end = a.end if a.end and a.end >= start else start + 149\n        end = min(end, start + 149)\n        path = os.path.join(ws, a.file)\n        if not os.path.isfile(path):\n            print(f\"(no such file: {a.file})\")\n        else:\n            lines = open(path, encoding=\"utf-8\", errors=\"replace\").read().split(\"\\n\")\n            print(\"\\n\".join(f\"{i:5}| {lines[i - 1]}\" for i in range(start, min(end, len(lines)) + 1)))\n        print(footer(used))\n    elif a.mode == \"history\":\n        if not (a.query or a.commit):\n            sys.exit('scout history needs args={\"mode\": \"history\", \"query\": \"NAME\"} or {\"mode\": \"history\", \"commit\": \"SHA\"}')\n        used = reading_budget()\n        if a.commit:\n            sha = a.commit.replace(\"'\", \"\")\n            print(run(f\"git show --stat --format='%h %s' '{sha}' | head -20 && git show --format= '{sha}' | head -80\", ws))\n        else:\n            q = a.query.replace(\"'\", \"'\\\\''\")\n            print(run(f\"git log --oneline -S'{q}' | head -15\", ws) or \"(no commit adds or removes it)\")\n            print(\"recent commits:\\n\" + run(\"git log --oneline -8\", ws))\n        print(footer(used))\n    elif a.mode == \"plan\":\n        if not a.text:\n            sys.exit('scout plan needs args={\"mode\": \"plan\", \"text\": \"FILE: ...\\\\nCHANGE: ...\"}')\n        with open(PLAN, \"w\", encoding=\"utf-8\") as f:\n            f.write(a.text.strip() + \"\\n\")\n        print(f\"PLAN SAVED to /tmp/plan.md ({len(a.text)} chars). Now call the editor.\")\n    else:  # check\n        diff = run(\"git diff --stat && git diff | head -80\", ws)\n        print(\"CURRENT CHANGES:\\n\" + (diff or \"(no changes)\"))\n        if a.test:\n            k = f\" -k '{a.k}'\" if a.k else \"\"\n            print(\"\\nTEST:\\n\" + run(f\"{sys.executable} -m pytest {a.test} -x -q -p no:cacheprovider{k} 2>&1 | tail -25\", ws, 200))\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "sub_agents/editor.yaml": "name: editor\ndescription: >-\n  Applies the plan saved in /tmp/plan.md to the source files. Request: one line, \"Apply /tmp/plan.md\".\n  Answers one line per file: EDITED (path and what changed) or FAILED (path and why).\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include ../prompts/editor.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\ngenerate_content_config: !include ../configs/editor_sampling.yaml\n"
 },
 "lo2": {
  "agent.yaml": "# Arm lo2 (rung-3f control for le2): lo with the fixed scout parser. lo: the same locator, scout skill (16 counted reading calls), settings and budget,\n# but no editor: it edits itself with edit_file. le vs lo separates \"the reading cap helps\" from \"the editor helps\".\nname: locator\ndescription: Finds what to change through the scout skill, edits it, checks and submits.\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/locator.md\ntools:\n  - get_status\n  - submit_patch\n  - edit_file\nskills:\n  - skills/scout\ngenerate_content_config: !include configs/locator_sampling.yaml\n",
  "configs/locator_sampling.yaml": "# Locator (rung-3e le): low temperature + deep thinking (budget to be set from rung-3d: 4096 / 2048 / 1024).\ntemperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  include_thoughts: true\n  thinking_budget: 2048\n",
  "eval_config.yaml": "# Arm lo2 (rung-3f): 36 calls, 8 min, 90 turns (same as le for a fair comparison).\nevaluation:\n  timeout_seconds: 240\n  max_tool_calls: 36\n  max_time_minutes: 8\n  max_turns: 90\n",
  "prompts/locator.md": "You are fixing one task in the repository at /workspace. You find what must change, make the edit with\nedit_file, and check the result. Nobody will answer questions: decide from the task text, the code, the docs\nand the tests.\n\n## The task (verbatim)\n{problem_description}\n\n## Your tools\n- scout is your only way to read code. Call run_skill_script with skill_name scout, file_path scripts/scout.py and:\n  find:  args {\"mode\": \"find\", \"query\": \"exact name or message\"}   search + the code that follows each hit\n  show:  args {\"mode\": \"show\", \"file\": \"pkg/module.py\", \"start\": 40, \"end\": 170}   up to 150 numbered lines\n  history: args {\"mode\": \"history\", \"query\": \"NAME\"}   commits that added or removed NAME (useful when the task\n         is only a title or a link), or {\"mode\": \"history\", \"commit\": \"abc1234\"} for one commit's change\n  check: args {\"mode\": \"check\", \"test\": \"tests/test_x.py\", \"k\": \"name\"}   current diff + that test's result\n  find, show and history share 16 reading calls; check is free.\n- edit_file makes the change: old_string = the exact current lines copied from show without the line-number\n  prefix, 1 to 5 consecutive lines with their indentation; new_string = the new lines.\n- submit_patch when the change is in.\n\n## Steps\n1. From the task, note the exact names, parameters, error messages, status codes and output it asks for. The\n   hidden tests use exactly these.\n2. Read: find the most specific name or message, then show whole functions. For a new feature, read the closest\n   existing sibling feature and copy its shape (signature, imports, exports, registration). For a documentation\n   example task, look under docs_src/ and docs/.\n3. As soon as you know the change (at the latest when scout says no reading calls are left), make the edit\n   with edit_file, using exactly the names and messages from step 1. Several small edits beat one large one. If\n   edit_file returns an error, never repeat the same call: show the exact lines once and rebuild old_string.\n4. Check with the nearest test file. Adjust the edit once only if the diff is not what you meant or a test fails\n   because of an error in the new code. A failing test that asserts the old behaviour the task asks to change is\n   expected: keep the change. Then submit_patch.\n\n## Rules\n- When a tool result says only a few tool calls remain: if no edit is in yet, make your best edit now; if it is\n  in, call submit_patch.\n- Call tools only through the tool-calling interface; never write a tool call or its output as text.\n- Never ask to change tests; the grader resets them.\n",
  "skills/scout/SKILL.md": "---\nname: scout\ndescription: The locator's reading tool. find, show and history read the repository (16 reading calls in total); plan writes /tmp/plan.md for the editor; check shows the current diff and runs a targeted test.\n---\n\nCall it with run_skill_script, skill_name scout, file_path scripts/scout.py and named args:\n\n- find:  args = {\"mode\": \"find\", \"query\": \"convert_underscores\"}\n  git grep hits for the name or message, each with the 20 lines that follow.\n- show:  args = {\"mode\": \"show\", \"file\": \"pkg/module.py\", \"start\": 40, \"end\": 170}\n  numbered lines, up to 150 at once.\n- history: args = {\"mode\": \"history\", \"query\": \"NAME\"} (commits that added or removed it, plus the 8 latest)\n  or {\"mode\": \"history\", \"commit\": \"abc1234\"} (that commit's change).\n- plan:  args = {\"mode\": \"plan\", \"text\": \"FILE: ...\\nWHAT IT DOES: ...\\nCHANGE: ...\\nCONTRACT: ...\"}\n  saves the plan for the editor to /tmp/plan.md.\n- check: args = {\"mode\": \"check\", \"test\": \"tests/test_x.py\", \"k\": \"name\"}\n  the current git diff, plus the last lines of that test run (test and k are optional).\n\nfind, show and history share a budget of 16 reading calls; every result ends with the count. After 16 they refuse:\nwrite the plan and call the editor. plan and check never count.\n",
  "skills/scout/scripts/scout.py": "\"\"\"Scout: the locator's only way to read the repository, with a built-in reading budget.\n\nCalled through run_skill_script with named args, e.g. args={\"mode\": \"find\", \"query\": \"convert_underscores\"}.\nModes:\n  find  --query TEXT                     git grep hits with surrounding code (counts as a reading call)\n  show  --file PATH --start N --end M    numbered lines, at most 150 (counts as a reading call)\n  plan  --text TEXT                      write the plan for the editor to /tmp/plan.md (free)\n  history --query TEXT | --commit SHA    commits that added/removed TEXT, or one commit's change (counts)\n  check [--test PATH] [--k EXPR]         git diff of the current changes + optional targeted pytest (free)\nAfter LIMIT reading calls, find/show refuse and tell the locator to write the plan and call the editor.\n\"\"\"\nimport argparse\nimport ast\nimport inspect\nimport json\nimport os\nimport subprocess\nimport sys\nimport tempfile\n\nLIMIT = 16\nWARN_FROM = 13\nTMP = tempfile.gettempdir()          # the task's /tmp (per-task directory in the subprocess sandbox)\nCOUNTER = os.path.join(TMP, \"scout_reads\")\nPLAN = os.path.join(TMP, \"plan.md\")\n\n\ndef workspace():\n    \"\"\"The repository root. The skill wrapper chdirs into a temp dir, so look for the directory it started in.\"\"\"\n    for frame in inspect.stack():\n        cwd = frame.frame.f_locals.get(\"_orig_cwd\")\n        if isinstance(cwd, str) and os.path.isdir(os.path.join(cwd, \".git\")):\n            return cwd\n    for cand in (os.environ.get(\"PYTHONPATH\", \"\").split(os.pathsep)[0], \"/workspace\"):\n        if cand and os.path.isdir(os.path.join(cand, \".git\")):\n            return cand\n    sys.exit(\"scout: repository not found\")\n\n\ndef run(cmd, cwd, timeout=120):\n    try:\n        res = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True, timeout=timeout)\n        return (res.stdout + res.stderr).rstrip()\n    except subprocess.TimeoutExpired:\n        return f\"(timed out after {timeout}s)\"\n\n\ndef reading_budget():\n    \"\"\"Increment and return the number of reading calls, or refuse when the budget is spent.\"\"\"\n    used = int(open(COUNTER).read() or 0) if os.path.exists(COUNTER) else 0\n    if used >= LIMIT:\n        print(f\"READING LIMIT REACHED ({LIMIT} reading calls used). Write your plan now with mode plan, then call the \"\n              \"editor. You can still use mode check after the editor has answered.\")\n        sys.exit(0)\n    used += 1\n    with open(COUNTER, \"w\") as f:\n        f.write(str(used))\n    return used\n\n\ndef footer(used):\n    left = LIMIT - used\n    msg = f\"[reading {used} of {LIMIT} used\"\n    if used >= WARN_FROM:\n        msg += f\"; {left} left: start writing the plan\" if left else \"; none left: write the plan now\"\n    return msg + \"]\"\n\n\nMODES = (\"find\", \"show\", \"history\", \"plan\", \"check\")\nALIASES = {\"path\": \"file\", \"filename\": \"file\", \"file_path\": \"file\", \"pattern\": \"query\", \"name\": \"query\",\n           \"q\": \"query\", \"search\": \"query\", \"line\": \"start\", \"start_line\": \"start\", \"from\": \"start\",\n           \"end_line\": \"end\", \"to\": \"end\", \"sha\": \"commit\", \"test_file\": \"test\", \"content\": \"text\", \"plan\": \"text\"}\n\n\ndef literal(text):\n    \"\"\"A dict or list written as JSON or as a Python repr (ADK passes a nested dict as str(dict)), else None.\"\"\"\n    text = text.strip()\n    if not text[:1] in \"{[\":\n        return None\n    for load in (json.loads, ast.literal_eval):\n        try:\n            return load(text)\n        except (ValueError, SyntaxError):\n            pass\n    return None\n\n\ndef as_int(value):\n    try:\n        return int(float(str(value).strip()))\n    except ValueError:\n        return None\n\n\ndef parse(argv):\n    \"\"\"Read every argument shape the model produces: {\"mode\": ...} (--mode X), a nested {\"args\": {...}}, a list of\n    [\"--mode\", \"show\", ...] or [\"--mode=show\"], positional [\"show\", \"pkg/a.py\", \"40\", \"160\"], \"file:line\" and \"40-160\".\n    Missing or zero line numbers get defaults instead of an error.\"\"\"\n    opts, pos, tokens = {}, [], list(argv)\n    while tokens:\n        tok = tokens.pop(0)\n        if tok == \"--\":\n            continue\n        val = literal(tok)\n        if isinstance(val, dict):\n            opts.update({str(k).lstrip(\"-\"): v for k, v in val.items()})\n        elif isinstance(val, list):\n            tokens = [str(v) for v in val] + tokens\n        elif tok.startswith(\"--\") or (tok.startswith(\"-\") and len(tok) > 1 and not tok[1].isdigit()):\n            key, eq, value = tok.lstrip(\"-\").partition(\"=\")\n            if not eq:\n                value = tokens.pop(0) if tokens and not tokens[0].startswith(\"--\") else \"\"\n            inner = literal(value) if key == \"args\" else None\n            if isinstance(inner, dict):\n                opts.update({str(k).lstrip(\"-\"): v for k, v in inner.items()})\n            elif isinstance(inner, list):\n                tokens = [str(v) for v in inner] + tokens\n            else:\n                opts[key] = value\n        else:\n            pos.append(tok)\n    opts = {ALIASES.get(k, k): v for k, v in opts.items()}\n    if \"args\" in opts and isinstance(opts[\"args\"], dict):\n        opts.update(opts.pop(\"args\"))\n    mode = str(opts.get(\"mode\", \"\")).strip().lower()\n    if mode not in MODES and pos and pos[0].lower() in MODES:\n        mode = pos.pop(0).lower()\n    if mode not in MODES:  # infer the mode from what was given\n        mode = \"show\" if opts.get(\"file\") else \"plan\" if opts.get(\"text\") else \"history\" if opts.get(\"commit\") else \\\n            \"find\" if opts.get(\"query\") else \"check\" if opts.get(\"test\") or opts.get(\"k\") else \"\"\n    a = argparse.Namespace(mode=mode, **{k: opts.get(k) for k in (\"query\", \"file\", \"start\", \"end\", \"text\",\n                                                                    \"commit\", \"test\", \"k\")})\n    if mode == \"show\":\n        if not a.file and pos:\n            a.file = pos.pop(0)\n        if pos and a.start is None:\n            a.start = pos.pop(0)\n        if pos and a.end is None:\n            a.end = pos.pop(0)\n        if a.file and \":\" in str(a.file):  # pkg/a.py:120 or pkg/a.py:100-160\n            a.file, _, where = str(a.file).partition(\":\")\n            a.start = a.start if a.start is not None else where\n        if a.start is not None and \"-\" in str(a.start).strip(\"-\") and a.end is None:\n            a.start, _, a.end = str(a.start).partition(\"-\")\n        a.start, a.end = as_int(a.start) if a.start is not None else None, as_int(a.end) if a.end is not None else None\n        if a.file:\n            a.file = str(a.file).strip().removeprefix(\"/workspace/\").removeprefix(\"./\")\n    elif mode in (\"find\", \"history\"):\n        if not a.query and not a.commit and pos:\n            a.query = \" \".join(pos)\n        if mode == \"history\" and a.query and not a.commit and all(c in \"0123456789abcdef\" for c in a.query) \\\n                and 7 <= len(a.query) <= 40:\n            a.commit, a.query = a.query, None\n    elif mode == \"plan\":\n        if not a.text and pos:\n            a.text = \" \".join(pos)\n    elif mode == \"check\":\n        if not a.test and pos:\n            a.test = pos.pop(0)\n    return a\n\n\ndef main():\n    a = parse(sys.argv[1:])\n    if not a.mode:\n        sys.exit('scout needs a mode: args={\"mode\": \"find\", \"query\": \"NAME\"} or {\"mode\": \"show\", \"file\": \"pkg/a.py\", '\n                 '\"start\": 1, \"end\": 150}; also history, plan, check')\n    ws = workspace()\n\n    if a.mode == \"find\":\n        if not a.query:\n            sys.exit('scout find needs args={\"mode\": \"find\", \"query\": \"NAME or message\"}')\n        used = reading_budget()\n        q = a.query.replace(\"'\", \"'\\\\''\")\n        out = run(f\"git grep -n -I -A 20 -e '{q}' -- '*.py' '*.md' | head -80\", ws)\n        if not out:\n            out = run(f\"git grep -n -I -i -e '{q}' -- . | head -30\", ws) or \"(no match)\"\n        print(out)\n        print(footer(used))\n    elif a.mode == \"show\":\n        if not a.file:\n            sys.exit('scout show needs args={\"mode\": \"show\", \"file\": \"path.py\", \"start\": 40, \"end\": 160}')\n        used = reading_budget()\n        start = max(a.start or 1, 1)\n        end = a.end if a.end and a.end >= start else start + 149\n        end = min(end, start + 149)\n        path = os.path.join(ws, a.file)\n        if not os.path.isfile(path):\n            print(f\"(no such file: {a.file})\")\n        else:\n            lines = open(path, encoding=\"utf-8\", errors=\"replace\").read().split(\"\\n\")\n            print(\"\\n\".join(f\"{i:5}| {lines[i - 1]}\" for i in range(start, min(end, len(lines)) + 1)))\n        print(footer(used))\n    elif a.mode == \"history\":\n        if not (a.query or a.commit):\n            sys.exit('scout history needs args={\"mode\": \"history\", \"query\": \"NAME\"} or {\"mode\": \"history\", \"commit\": \"SHA\"}')\n        used = reading_budget()\n        if a.commit:\n            sha = a.commit.replace(\"'\", \"\")\n            print(run(f\"git show --stat --format='%h %s' '{sha}' | head -20 && git show --format= '{sha}' | head -80\", ws))\n        else:\n            q = a.query.replace(\"'\", \"'\\\\''\")\n            print(run(f\"git log --oneline -S'{q}' | head -15\", ws) or \"(no commit adds or removes it)\")\n            print(\"recent commits:\\n\" + run(\"git log --oneline -8\", ws))\n        print(footer(used))\n    elif a.mode == \"plan\":\n        if not a.text:\n            sys.exit('scout plan needs args={\"mode\": \"plan\", \"text\": \"FILE: ...\\\\nCHANGE: ...\"}')\n        with open(PLAN, \"w\", encoding=\"utf-8\") as f:\n            f.write(a.text.strip() + \"\\n\")\n        print(f\"PLAN SAVED to /tmp/plan.md ({len(a.text)} chars). Now call the editor.\")\n    else:  # check\n        diff = run(\"git diff --stat && git diff | head -80\", ws)\n        print(\"CURRENT CHANGES:\\n\" + (diff or \"(no changes)\"))\n        if a.test:\n            k = f\" -k '{a.k}'\" if a.k else \"\"\n            print(\"\\nTEST:\\n\" + run(f\"{sys.executable} -m pytest {a.test} -x -q -p no:cacheprovider{k} 2>&1 | tail -25\", ws, 200))\n\n\nif __name__ == \"__main__\":\n    main()\n"
 }
}

shutil.rmtree(SUB_ROOT, ignore_errors=True)
for name, files in EMBEDDED.items():
    for rel, text in files.items():
        path = SUB_ROOT / name / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding='utf-8')

if 'sample' in CONFIGS:
    # Google's sample as shipped, minus its LoRA adapters.
    sample = SUB_ROOT / 'sample'
    shutil.copytree(DATA_DIR / 'sample_submission', sample)
    shutil.rmtree(sample / 'adapters', ignore_errors=True)
    for yml in list(sample.rglob('*.yaml')):
        yml.write_text(re.sub(r'^adapter:.*\n', '', yml.read_text(), flags=re.M))
    (sample / 'eval_config.yaml').write_text(
        'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 100\n  max_time_minutes: 5.0\n  max_turns: 90\n')

for name in CONFIGS:
    print(name, sorted(p.relative_to(SUB_ROOT / name).as_posix() for p in (SUB_ROOT / name).rglob('*') if p.is_file()))

In [ ]:
# 3. Start vLLM once for all configs (no adapters → LoRA off → more KV cache). Takes ~6–8 min.
import litellm, torch
from adk_submission import VllmConfig, VllmServer

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
# The model must be attached as notebook input (Add Input -> Models -> google/gemma-4, variant TARGET_MODEL_NAME).
_cands = sorted(p for p in glob.glob(f'/kaggle/input/models/google/gemma-4/**/{TARGET_MODEL_NAME}/*', recursive=True)
                if Path(p, 'config.json').exists())
assert _cands, f'Model {TARGET_MODEL_NAME} not found under /kaggle/input: attach it as a notebook input'
MODEL_PATH = Path(_cands[-1])  # latest version
print('MODEL_PATH', MODEL_PATH)
# Per-model vLLM flags (host, forum 748287; the scorer has the same): the 12B checkpoint is labelled
# gemma4_unified, which vLLM 0.19.1 cannot load, so vLLM is told to load it as the regular gemma4 text model.
MODEL_VLLM_EXTRA_ARGS: dict[str, tuple[str, ...]] = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
# LOCAL TEST WORKAROUND (the scorer runs its own vLLM): with that flag, vLLM 0.19.1 (wheelhouse v30) still crashes
# on the 12B (KeyError 'vision_embedder.patch_dense.weight'). gemma4.py already skips the multimodal weights by name
# (audio_tower., vision_tower., embed_audio., embed_vision.), but the 12B file names 9 of its image weights
# model.vision_embedder.*, which is not in that list. Add it, nothing else: every other tensor name in the 12B file
# (checked from its safetensors header: model.language_model.*, 48 standard gemma4 layers) loads as before, and any
# other unknown name would still fail loudly.
if TARGET_MODEL_NAME == 'gemma-4-12b-it-qat-w4a16-ct':
    import importlib.util  # locate the file without importing vLLM's model code in this process
    _src_path = Path(importlib.util.find_spec('vllm').origin).parent / 'model_executor' / 'models' / 'gemma4.py'
    _src = _src_path.read_text()
    _old = '            "embed_vision.",\n'
    if 'SKIP_VISION_EMBEDDER' not in _src:
        assert _src.count(_old) == 1, 'gemma4.py skip list not found: vLLM changed, do not run the 12B'
        _src_path.write_text(_src.replace(_old, _old + '            "vision_embedder.",  # SKIP_VISION_EMBEDDER\n'))
    print('gemma4.py skip list now includes vision_embedder.')
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=list(MODEL_VLLM_EXTRA_ARGS.get(TARGET_MODEL_NAME, ())),
)
server_instance = VllmServer(vllm_cfg)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
models = server_instance.create_model_registry(aliases=[TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')

# Installed harness versions (the wheelhouse changes over time; rung-1i notes).
from importlib.metadata import version, PackageNotFoundError
def _ver(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return None
print('VERSIONS', {p: _ver(p) for p in ('adk-submission', 'swegemma', 'adk-eval-core', 'google-adk', 'vllm')})

# Sampling diagnostic (rung-1j): replay the REAL first orchestrator request (full 1b prompt + harness task message)
# under several sampling settings, thinking off. Counts hidden reasoning, thought markers leaking into text, and
# whether a proper tool call came back. Task text is read at runtime from the competition data (never committed).
def _sampling_diagnostic(task_ids=("rich_3882", "fastapi_15589"), samples=3):
    import json as _json, time, types, urllib.request as _url
    from swegemma.harness.agent_runner import build_agent_prompt
    _base = server_instance.base_url.rstrip("/")
    _base = _base if _base.endswith("/v1") else _base + "/v1"
    _served = _json.load(_url.urlopen(_base + "/models"))["data"][0]["id"]
    def _fn(name, desc, props=None, req=()):
        return {"type": "function", "function": {"name": name, "description": desc, "parameters":
                {"type": "object", "properties": props or {}, "required": list(req)}}}
    _s = {"type": "string"}; _i = {"type": "integer"}
    _tools = [_fn("run_command", "Run a shell command in the workspace.", {"command": _s}, ["command"]),
              _fn("read_file", "Read a file.", {"filepath": _s, "start_line": _i, "end_line": _i}, ["filepath"]),
              _fn("get_status", "Budget and progress."), _fn("submit_patch", "Submit the current diff."),
              _fn("search_similar_code", "Semantic code search.", {"query": _s}, ["query"]),
              _fn("get_code_neighbors", "Graph neighbors of a symbol.", {"symbol": _s}, ["symbol"]),
              _fn("get_code_subgraph", "Graph around a symbol.", {"symbol": _s}, ["symbol"]),
              _fn("executor", "Makes the code change.", {"request": _s}, ["request"]),
              _fn("verifier", "Checks the change.", {"request": _s}, ["request"])]
    _tasks = {}
    for _line in (DATA_DIR / "tasks.jsonl").read_text().splitlines():
        _d = _json.loads(_line)
        if _d.get("instance_id") in task_ids:
            _tasks[_d["instance_id"]] = types.SimpleNamespace(**_d)
    class _NS(types.SimpleNamespace):
        def __getattr__(self, _name):  # any field the harness reads but we did not set -> None
            return None
    _cfg = _NS(budget=_NS(time_minutes=5.0, tool_calls=80, turns=90), harness=_NS(command_timeout_seconds=240, max_stdout_chars=5000),
               enable_sandbox_testing=True)
    _instr = (SUB_ROOT / "1b" / "prompts" / "orchestrator.md").read_text()
    _settings = [("T1.0 p0.95 k64", 1.0, 0.95, 64), ("T1.0 p0.80 k20", 1.0, 0.80, 20), ("T0.6 p0.95 k64", 0.6, 0.95, 64),
                 ("T0.4 p0.95 k64", 0.4, 0.95, 64), ("T0.2 p0.95 k64", 0.2, 0.95, 64)]
    _summary = {}
    for _tid, _task in _tasks.items():
        try:
            _user = build_agent_prompt(_task, _cfg, "", declared_tools=[t["function"]["name"] for t in _tools])
        except Exception as _e:
            print("build_agent_prompt failed, using plain task text:", repr(_e))
            _user = "Problem Statement:\n" + _task.problem_statement
        _msgs = [{"role": "system", "content": _instr.replace("{problem_description}", _task.problem_statement)},
                 {"role": "user", "content": _user}]
        for _name, _t, _p, _k in _settings:
            for _rep in range(samples):
                _body = {"model": _served, "messages": _msgs, "tools": _tools, "temperature": _t, "top_p": _p,
                         "top_k": _k, "max_tokens": 1024, "chat_template_kwargs": {"enable_thinking": False}}
                _t0 = time.time()
                _req = _url.Request(_base + "/chat/completions", data=_json.dumps(_body).encode(),
                                    headers={"Content-Type": "application/json"})
                _r = _json.load(_url.urlopen(_req, timeout=600))
                _m = _r["choices"][0]["message"]
                _reason = _m.get("reasoning_content") or _m.get("reasoning") or ""
                _content = _m.get("content") or ""
                _ok = bool(_m.get("tool_calls"))
                _leak = "<|channel>" in _content or "<channel|>" in _content
                _row = (time.time() - _t0, _r["usage"]["completion_tokens"], len(_reason), _leak, _ok)
                _summary.setdefault(_name, []).append(_row)
                print(f"{_tid:14} {_name}  #{_rep + 1}: {_row[0]:5.1f}s  out_tok={_row[1]:5}  reasoning_chars={_row[2]:5}  "
                      f"marker_in_text={_leak}  tool_call={_ok}  prompt_tok={_r['usage']['prompt_tokens']}")
                if (_reason or _leak) and _rep == 0:
                    print("    starts:", repr((_reason or _content)[:160]))
    print("SAMPLING SUMMARY (per setting over all tasks and samples)")
    for _name, _rows in _summary.items():
        _n = len(_rows)
        print(f"  {_name}: mean {sum(r[0] for r in _rows) / _n:5.1f}s  mean out_tok {sum(r[1] for r in _rows) / _n:6.0f}  "
              f"with reasoning {sum(r[2] > 0 for r in _rows)}/{_n}  marker leaks {sum(r[3] for r in _rows)}/{_n}  "
              f"proper tool calls {sum(r[4] for r in _rows)}/{_n}")
RUN_SAMPLING_DIAGNOSTIC = False  # rung-1k answered it: direct calls never think
try:
    if RUN_SAMPLING_DIAGNOSTIC:
        _sampling_diagnostic()
except Exception as _e:  # never block the evaluation
    print("sampling diagnostic failed:", repr(_e))

In [ ]:
# 4. Evaluate every config on the same tasks (Evaluator configured like the scorer; see scripts/run_eval.py)
import asyncio, concurrent.futures, time, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

DATA = DATA_DIR  # default used by build_config (copied from scripts/run_eval.py)


def read_eval_config(submission_dir):
    """The four eval_config.yaml keys the scorer reads (defaults as in the host notebook)."""
    path = Path(submission_dir) / "eval_config.yaml"
    raw = yaml.safe_load(path.read_text()) if path.exists() else {}
    section = (raw or {}).get("evaluation", raw or {})
    return {
        "timeout_seconds": int(section.get("timeout_seconds", 300)),
        "max_tool_calls": section.get("max_tool_calls"),
        "max_time_minutes": float(section.get("max_time_minutes", 5.0)),
        "max_turns": section.get("max_turns"),
    }


def wheels_dir_for(data_dir):
    """Official test wheels, plus data/wheels_extra if present (local only: test deps missing from the public set,
    forum 745775). Merged as links into data/wheels_merged so the official folder stays untouched."""
    official, extra = Path(data_dir) / "wheels", Path(data_dir) / "wheels_extra"
    if not extra.is_dir():
        return official
    merged = Path(data_dir) / "wheels_merged"
    merged.mkdir(exist_ok=True)
    for wheel in [*official.glob("*.whl"), *extra.glob("*.whl")]:
        link = merged / wheel.name
        if not link.exists():
            link.hardlink_to(wheel)
    return merged


def build_config(submission_dir, task_ids, results_dir, models, data_dir=DATA, sandbox="docker",
                 adapter_manifest=None, display_mode="single", concurrency=1):
    limits, gen_constraints = build_submission_limits()
    budgets = read_eval_config(submission_dir)
    return EvalConfig(
        tasks_path=Path(data_dir) / "tasks.jsonl",
        snapshots_dir=Path(data_dir) / "snapshots",
        results_dir=Path(results_dir),
        submission_dir=Path(submission_dir),
        models=models,
        sandbox=sandbox,
        wheels_dir=wheels_dir_for(data_dir),
        graph_dir=str(Path(data_dir) / "graphs"),
        embeddings_dir=str(Path(data_dir) / "embeddings"),
        task_ids=list(task_ids),
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapter_manifest,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=5, overlap_size=2, token_threshold=14336, event_retention_size=5,
        ),
        display_mode=display_mode,
        concurrency=concurrency,
        **budgets,
    )


def run_sync(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro)).result()
    return asyncio.run(coro)


RESULTS = WORKING_DIR / 'results'
shutil.rmtree(RESULTS, ignore_errors=True)
# Repeats are interleaved (r1 of every config, then r2, ...) so a session that dies early still has complete pairs.
RUNS = [(name, f'{name}_r{rep + 1}' if REPEATS > 1 else name) for rep in range(REPEATS) for name in CONFIGS]
wall = {}
for name, key in RUNS:
    start = time.time()
    config = build_config(SUB_ROOT / name, TASK_IDS, RESULTS / key, models, data_dir=DATA_DIR,
                          sandbox='subprocess', display_mode='single' if CONCURRENCY == 1 else 'quiet',
                          concurrency=CONCURRENCY)
    result = run_sync(Evaluator(config).run())
    wall[key] = time.time() - start
    print(f'==> {key}: {result.resolved}/{result.total} resolved in {wall[key] / 60:.1f} min')
    shutil.make_archive(str(WORKING_DIR / 'results'), 'zip', RESULTS)  # partial results survive a crash

In [ ]:
# 5. Comparison table + results.zip (download it from the Output panel), then STOP THE SESSION.
import json
from collections import Counter


NUDGE_PREFIXES = ("Please continue your work", "Your previous response reached the token limit")


HELPER_FORMATS = {"reader": "LOCATION:", "executor": "CHANGED:", "verifier": "VERDICT:"}


TEST_PATHS = ("tests/", "test_", "conftest.py", "pytest.ini")


def bucket(task, patch=""):
    """Coarse failure bucket from the harness result (patch text comes from patches/<id>.patch)."""
    error = (task.get("error") or "").lower()
    patch = patch or task.get("agent_patch") or ""
    if task.get("resolved"):
        return "resolved"
    if "context" in error and ("window" in error or "length" in error):
        return "overflow"
    if "not found" in error and "tool" in error:
        return "unknown tool"
    if not patch.strip():
        return "timeout, no patch" if "timeout" in error or "timed out" in error else "no patch"
    changed = [line.split(" b/")[-1] for line in patch.splitlines() if line.startswith("diff --git")]
    if any(any(t in f for t in TEST_PATHS) for f in changed):
        return "edited tests"
    return "wrong fix"


def trace_stats(trace):
    steps = trace.get("steps", [])
    stats = Counter()
    tools = Counter()
    commands = Counter()
    helper_ok = Counter()
    helper_total = Counter()
    last_prompt = {}  # per agent: helpers have their own conversations
    peak_prompt = 0
    for step in steps:
        message = str(step.get("message") or "")
        if step.get("source") != "agent" and message.startswith(NUDGE_PREFIXES):
            stats["nudges"] += 1
        metrics = step.get("metrics") or {}
        prompt = metrics.get("prompt_tokens")
        if prompt:
            stats["model_calls"] += 1
            stats["output_tokens"] += metrics.get("completion_tokens") or 0
            peak_prompt = max(peak_prompt, prompt)
            # Compaction replaces old events with a summary: the agent's next prompt is much smaller than its last.
            # A helper (agent_tool) starts a fresh conversation on every call, so a new call resets its baseline.
            author = (step.get("extra") or {}).get("author")
            previous = last_prompt.get(author)
            if previous and prompt < 0.7 * previous:
                stats["compactions"] += 1
            last_prompt[author] = prompt
        for call in step.get("tool_calls") or []:
            name = call.get("function_name")
            tools[name] += 1
            if name == "run_command":
                commands[(call.get("arguments") or {}).get("command", "")] += 1
            if name in HELPER_FORMATS:
                last_prompt.pop(name, None)
        # Observations are tagged with the tool that produced them. Helper (agent_tool) replies arrive as
        # {"raw": "<final text>"} in the step whose observation is tagged with the helper's name.
        observation = step.get("observation") or {}
        source = (observation.get("extra") or {}).get("tool_name")
        content = str(observation.get("content", ""))
        if '"status": "error"' in content[:40] or "'status': 'error'" in content[:40] or content.startswith('{"error"'):
            stats["tool_errors"] += 1
        if source in HELPER_FORMATS:
            helper_total[source] += 1
            try:
                reply = json.loads(content).get("raw", content)
            except (ValueError, AttributeError):
                reply = content
            if HELPER_FORMATS[source] in reply[:300]:
                helper_ok[source] += 1
            if "<|tool_call>" in reply:
                stats["broken_calls"] += 1
        if "<|tool_call>" in message:
            stats["broken_calls"] += 1
    stats["peak_prompt"] = peak_prompt
    stats["repeated_cmds"] = sum(n - 1 for n in commands.values() if n > 1)
    helpers = {h: f"{helper_ok[h]}/{helper_total[h]}" for h in helper_total}
    return stats, tools, helpers


def analyze(results_dir):
    results_dir = Path(results_dir)
    tasks = [json.loads(line) for line in (results_dir / "task_results.jsonl").read_text().splitlines() if line]
    print(f"\n## {results_dir.name}: {sum(t.get('resolved', False) for t in tasks)}/{len(tasks)} resolved")
    header = (f"{'task':<16} {'outcome':<18} {'sec':>5} {'calls':>5} {'out_tok':>7} {'peak_in':>7} "
              f"{'compact':>7} {'nudge':>5} {'repeat':>6} {'errors':>6} {'broken':>6}  tools / helpers")
    print(header)
    print("-" * len(header))
    for task in tasks:
        task_id = task.get("instance_id") or task.get("task_id")
        trace_file = results_dir / "traces" / f"trace_{task_id}.json"
        trace = json.loads(trace_file.read_text()) if trace_file.exists() else {}
        stats, tools, helpers = trace_stats(trace)
        patch_file = results_dir / "patches" / f"{task_id}.patch"
        patch = patch_file.read_text() if patch_file.exists() else ""
        tool_list = ", ".join(f"{name}×{n}" for name, n in tools.most_common())
        helper_list = (" | format ok " + ", ".join(f"{h} {v}" for h, v in helpers.items())) if helpers else ""
        print(f"{task_id:<16} {bucket(task, patch):<18} {task.get('duration_seconds', 0):>5.0f} {stats['model_calls']:>5} "
              f"{stats['output_tokens']:>7} {stats['peak_prompt']:>7} {stats['compactions']:>7} "
              f"{stats['nudges']:>5} {stats['repeated_cmds']:>6} {stats['tool_errors']:>6} {stats['broken_calls']:>6}  "
              f"{tool_list}{helper_list}")
        if patch:
            changed = [line.split(" b/")[-1] for line in patch.splitlines() if line.startswith("diff --git")]
            print(f"{'':<16} patch: {', '.join(changed)}")
        if task.get("error"):
            print(f"{'':<16} error: {task['error'][:150]}")


for _, key in RUNS:
    analyze(RESULTS / key)
shutil.make_archive(str(WORKING_DIR / 'results'), 'zip', RESULTS)
print('\nWall clock per config (min):', {k: round(v / 60, 1) for k, v in wall.items()})
print('Wrote /kaggle/working/results.zip')